# Build a SQL agent

## Overview

In this tutorial, you will learn how to build an agent that can answer questions about a SQL database using LangChain agents.

At a high level, the agent will:

1. Fetch the available tables and schemas from the database
2. Decide which tables are relevant to the question
3. Fetch the schemas for the relevant tables
4. Generate a query based on the question and information from the schemas
5. Double-check the query for common mistakes using an LLM
6. Execute the query and return the results
7. Correct mistakes surfaced by the database engine until the query is successful
8. Formulate a response based on the results

::: {.callout-warning}
Building Q&A systems of SQL databases requires executing model-generated SQL queries. There are inherent risks in doing this. Make sure that your database connection permissions are always scoped as narrowly as possible for your agent’s needs. This will mitigate, though not eliminate, the risks of building a model-driven system.
:::

## Concepts

We will cover the following concepts:

- [Tools](https://docs.langchain.com/oss/python/langchain/tools) for reading from SQL databases
- LangChain [agents](https://docs.langchain.com/oss/python/langchain/agents)
- [Human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop) processesWe will cover the following concepts:

## Environment Setup

### Environment variables

The following environment variables are used in this notebook:

| Variable               | Required | Purpose                                  |
|------------------------|----------|------------------------------------------|
| `OPENROUTER_API_KEY`   | Yes      | Authenticates requests to OpenRouter LLMs |

#### On Colab?

Store these as Colab **Secrets** (key icon in the left sidebar), using the variable names above (for example `OPENROUTER_API_KEY`). Grant this notebook access to the secrets so they can be read securely at runtime.

#### Locally?

Store these in a `.env` file in your project directory (which should be listed in `.gitignore` so it is never committed). The code below uses `python-dotenv` to load values from this file into environment variables at runtime.


::: {.callout-warning}

API Keys are secrets and thus [**shall never be exposed**](./never_expose_api_keys.qmd).

:::

### Reading environment variables in

In [1]:
import os

In [2]:
try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    # Load `.env` file (locally)
    from dotenv import load_dotenv
    load_dotenv(override=True)

### Install dependencies

In [3]:
# %pip install langchain langgraph langchain-community

In [4]:
# Prettier printing for strings with structured data
from rich import print

## 1. Select an LLM

Select a model that supports [tool-calling](https://docs.langchain.com/oss/python/integrations/providers/overview). Note:

- We first used the `nvidia/nemotron-3-nano-30b-a3b:free` model but it couldn't finish the task.
- We then switched to `nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free` and it did.

In [5]:
from langchain_openai import ChatOpenAI


model_nemotron_3_nano_omni = ChatOpenAI(
    model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    temperature=0,
    # OpenRouter instead of the default OpenAI API
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

/home/halgoz/work/Bootcamp/ai-pros/public/content/W4/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configure the database

You will be creating a [SQLite database](https://www.sqlitetutorial.net/sqlite-sample-database/) for this tutorial. SQLite is a lightweight database that is easy to set up and use. We will be loading the `chinook` database, which is a sample database that represents a digital media store.

For convenience, we have hosted the database (`Chinook.db`) on a public GCS bucket.


In [6]:
import pathlib
import requests # third-party library

url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"
local_path = pathlib.Path("Chinook.db")

if local_path.exists():
    print(f"{local_path} already exists, skipping download.")
else:
    response = requests.get(url)
    if response.status_code == 200:
        local_path.write_bytes(response.content)
        print(f"File downloaded and saved as {local_path}")
    else:
        print(f"Failed to download the file. Status code: {response.status_code}")

Chinook.db already exists, skipping download.

We will use a handy SQL database wrapper available in the `langchain_community` package to interact with the database. The wrapper provides a simple interface to execute SQL queries and fetch results:


In [7]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

print(f"Dialect: {db.dialect}")
print(f"Tables: {db.get_usable_table_names()}")

query = "SELECT * FROM Artist LIMIT 5;"
print(f'Sample row: {db.run(query)}')

Dialect: sqlite

Tables: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 
'PlaylistTrack', 'Track']

Sample row: [(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]

## 3. Add tools for database interactions

Use the `SQLDatabase` wrapper available in the `langchain_community` package to interact with the database. The wrapper provides a simple interface to execute SQL queries and fetch results:

In [8]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=model_nemotron_3_nano_omni)

tools = toolkit.get_tools()

for tool in tools:
    print(f"{tool.name}: {tool.description}\n")

sql_db_query: Input to this tool is a detailed and correct SQL query, output is a result from the database. If the 
query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the 
query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to 
query the correct table fields.

sql_db_schema: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for 
those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, 
table2, table3

sql_db_list_tables: Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker: Use this tool to double check if your query is correct before executing it. Always use this 
tool before executing a query with sql_db_query!

## 4. Use `create_agent`

Use [`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_agent) to build a [ReAct agent](https://arxiv.org/pdf/2210.03629) with minimal code. The agent will interpret the request and generate a SQL command, which the tools will execute. If the command has an error, the error message is returned to the model. The model can then examine the original request and the new error message and generate a new command. This can continue until the LLM generates the command successfully or reaches an end count. This pattern of providing a model with feedback - error messages in this case - is very powerful.

Initialize the agent with a descriptive system prompt to customize its behavior:

In [9]:
system_prompt = """
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables.
""".format(
    dialect=db.dialect,
    top_k=5,
)

Now, create an agent with the model, tools, and prompt:


In [12]:
from langchain.agents import create_agent


agent = create_agent(
    model=model_nemotron_3_nano_omni,
    tools=tools,
    system_prompt=system_prompt,
)

## 5. Run the agent

Run the agent on a sample query and observe its behavior:

In [13]:
from langchain.messages import HumanMessage

question = "Which genre on average has the longest tracks?"
steps = []
for step in agent.stream(
    {"messages": [HumanMessage(question)]},
    stream_mode="values",
):
    steps.append(step)
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Which genre on average has the longest tracks?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_fc6c2b6df93c4d52ad6ed15e)
 Call ID: call_fc6c2b6df93c4d52ad6ed15e
  Args:
    tool_input:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_3effddc158834cb4bcb34512)
 Call ID: call_3effddc158834cb4bcb34512
  Args:
    table_names: Track, Genre
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE "Genre" (
	"GenreId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("GenreId")
)

/*
3 rows from Genre 

The agent correctly wrote a query, checked the query, and ran it to inform its final response.

How many messages were generated because of this invokation?

In [14]:
print(len(steps))

8

What are the last three?

1. `AIMessage` that calls to `sql_db_query`
2. `ToolMessage` with the result of executing the SQL query
3. `AIMessage` with the final answer: `Sci Fi & Fantasy`

In [15]:
print(steps[-3:])

[
    {
        'messages': [
            HumanMessage(
                content='Which genre on average has the longest tracks?',
                additional_kwargs={},
                response_metadata={},
                id='7051f9e3-8c0e-4a28-85b4-e10efcee4352'
            ),
            AIMessage(
                content='',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 372,
                        'prompt_tokens': 889,
                        'total_tokens': 1261,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': 0,
                            'reasoning_tokens': 389,
                            'rejected_prediction_tokens': None,
                            'image_tokens': 0
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': 0,
                            'cached_tokens': 0,
                            'cache_write_tokens': 0,
                            'video_tokens': 0
                        },
                        'cost': 0,
                        'is_byok': False,
                        'cost_details': {
                            'upstream_inference_cost': 0,
                            'upstream_inference_prompt_cost': 0,
                            'upstream_inference_completions_cost': 0
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'nvidia/nemotron-3-nano-omni-30b-a3b-reasoning-20260428:free',
                    'system_fingerprint': None,
                    'id': 'gen-1777615051-swzLepqE1XxaLAGUhdS5',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--019de21d-1832-7453-8f90-e689fb62e065-0',
                tool_calls=[
                    {
                        'name': 'sql_db_list_tables',
                        'args': {'tool_input': ''},
                        'id': 'call_fc6c2b6df93c4d52ad6ed15e',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 889,
                    'output_tokens': 372,
                    'total_tokens': 1261,
                    'input_token_details': {'audio': 0, 'cache_read': 0},
                    'output_token_details': {'audio': 0, 'reasoning': 389}
                }
            ),
            ToolMessage(
                content='Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, 
PlaylistTrack, Track',
                name='sql_db_list_tables',
                id='724b444f-179f-44f5-a114-fd981a556757',
                tool_call_id='call_fc6c2b6df93c4d52ad6ed15e'
            ),
            AIMessage(
                content='',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 431,
                        'prompt_tokens': 971,
                        'total_tokens': 1402,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': 0,
                            'reasoning_tokens': 463,
                            'rejected_prediction_tokens': None,
                            'image_tokens': 0
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': 0,
                            'cached_tokens': 0,
                            'cache_write_tokens': 0,
                            'video_tokens': 0
                        },
            

## 6. Implement human-in-the-loop review

It can be prudent to check the agent's SQL queries before they are executed for any unintended actions or inefficiencies.

LangChain agents feature support for built-in [human-in-the-loop middleware](https://docs.langchain.com/oss/python/langchain/human-in-the-loop) to add oversight to agent tool calls. Let's configure the agent to pause for human review on calling the `sql_db_query` tool:

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model_nemotron_3_nano_omni,
    tools=tools,
    system_prompt=system_prompt,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"sql_db_query": True},
            description_prefix="Tool execution pending approval",
        ),
    ],
    checkpointer=InMemorySaver(),
)

:::{.callout-note}
We've added a [checkpointer](https://docs.langchain.com/oss/python/langchain/short-term-memory) to our agent to allow execution to be paused and resumed. See the [human-in-the-loop guide](https://docs.langchain.com/oss/python/langchain/human-in-the-loop) for detalis on this as well as available middleware configurations.
:::

On running the agent, it will now pause for review before executing the `sql_db_query` tool:


In [17]:
config = {"configurable": {"thread_id": "1"}}
question = "Which genre on average has the longest tracks?"
for step in agent.stream(
    input={"messages": [HumanMessage(question)]},
    config=config,
    stream_mode="values",
):
    if "messages" in step:
        step["messages"][-1].pretty_print()

================================ Human Message =================================

Which genre on average has the longest tracks?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_11d3c9676ddf49f2a09fe7b2)
 Call ID: call_11d3c9676ddf49f2a09fe7b2
  Args:
    tool_input:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_04f6556fd432400da65c357d)
 Call ID: call_04f6556fd432400da65c357d
  Args:
    table_names: Genre
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE "Genre" (
	"GenreId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("GenreId")
)

/*
3 rows from Genre table:


In [18]:
if "__interrupt__" in step:
    print("INTERRUPTED:")
    interrupt = step["__interrupt__"][0]
    for request in interrupt.value["action_requests"]:
        print(request["description"])

INTERRUPTED:

Tool execution pending approval

Tool: sql_db_query
Args: {'query': 'SELECT g.Name AS Genre, AVG(t.Milliseconds) AS AvgMilliseconds FROM Track t JOIN Genre g ON 
t.GenreId = g.GenreId GROUP BY g.Name ORDER BY AvgMilliseconds DESC LIMIT 5;'}

We can resume execution, in this case accepting the query, using [Command](https://docs.langchain.com/oss/python/langgraph/use-graph-api#combine-control-flow-and-state-updates-with-command):

In [19]:
from langgraph.types import Command 

for step in agent.stream(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,
    stream_mode="values",
):
    if "messages" in step:
        step["messages"][-1].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  sql_db_query (call_e06c2e87b15f403e86f5035e)
 Call ID: call_e06c2e87b15f403e86f5035e
  Args:
    query: SELECT g.Name AS Genre, AVG(t.Milliseconds) AS AvgMilliseconds FROM Track t JOIN Genre g ON t.GenreId = g.GenreId GROUP BY g.Name ORDER BY AvgMilliseconds DESC LIMIT 5;
================================== Ai Message ==================================
Tool Calls:
  sql_db_query (call_e06c2e87b15f403e86f5035e)
 Call ID: call_e06c2e87b15f403e86f5035e
  Args:
    query: SELECT g.Name AS Genre, AVG(t.Milliseconds) AS AvgMilliseconds FROM Track t JOIN Genre g ON t.GenreId = g.GenreId GROUP BY g.Name ORDER BY AvgMilliseconds DESC LIMIT 5;
================================= Tool Message =================================
Name: sql_db_query

[('Sci Fi & Fantasy', 2911783.0384615385), ('Science Fiction', 2625549.076923077), ('Drama', 2575283.78125), ('TV Shows', 2145041.0215053763), ('Comedy', 1585263.7

In [24]:
if "__interrupt__" in step:
        print("INTERRUPTED:")
        interrupt = step["__interrupt__"][0]
        for request in interrupt.value["action_requests"]:
            print(request["description"])
else:
    print("Final Answer:\n", step["messages"][-1].content)

Final Answer:
 Sci Fi & Fantasy, with an average track length of approximately 2911783 milliseconds (approximately 55 minutes).

## Next steps

For deeper customization, check out [this tutorial](https://docs.langchain.com/oss/python/langgraph/sql-agent) for implementing a SQL agent directly using LangGraph primitives.

## References

- [Build a SQL agent](https://docs.langchain.com/oss/python/langchain/sql-agent)